# MNIST Digit Identification using an I-JEPA Approach
This notebook implements a Joint-Embedding Predictive Architecture (JEPA) for MNIST.
Instead of predicting missing pixels, it predicts the latent representations of missing patches.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import math
import copy

In [7]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Hyperparameters & Configuration

In [8]:
# Data configuration
BATCH_SIZE = 64
PATCH_SIZE = 4       # MNIST 28x28 split into 4x4 patches -> 7x7 grid (49 patches)
IMG_SIZE = 28
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2

# Architecture configuration
EMBED_DIM = 64
NUM_HEADS = 4
DEPTH = 3
PREDICTOR_DEPTH = 2

# Training configuration
EPOCHS = 5
LR = 1e-3
EMA_DECAY = 0.996  # Momentum for Target Encoder update

## 2. Dataset Setup
We load standard MNIST. The self-supervised phase ignores the labels.

In [9]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 3. Model Architecture
We define the Patch Embedding, the Transformer Block, and the Context/Target/Predictor modules.


In [10]:

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size=4, in_chans=1, embed_dim=64):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # [B, 1, 28, 28] -> [B, embed_dim, 7, 7] -> [B, 49, embed_dim]
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class TransformerLayer(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, x):
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x

In [11]:


class JEPAEncoder(nn.Module):
    def __init__(self, num_patches=49, embed_dim=64, heads=4, depth=3):
        super().__init__()
        self.patch_embed = PatchEmbedding(embed_dim=embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.blocks = nn.ModuleList([TransformerLayer(embed_dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        
        # Initialize weights
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x, patch_indices=None):
        # If patch_indices is given, we filter to keep only those specific patches (Context)
        x = self.patch_embed(x)
        x = x + self.pos_embed
        
        if patch_indices is not None:
            # Gather matching patches per batch item
            batch_size = x.shape[0]
            idx = patch_indices.unsqueeze(-1).expand(-1, -1, x.shape[-1])
            x = torch.gather(x, dim=1, index=idx)
            
        for block in self.blocks:
            x = block(x)
        return self.norm(x)

In [12]:


class JEPAPredictor(nn.Module):
    def __init__(self, embed_dim=64, heads=4, depth=2, num_patches=49):
        super().__init__()
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.blocks = nn.ModuleList([TransformerLayer(embed_dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, context_feats, context_idx, target_idx):
        B = context_feats.shape[0]
        
        # Create a full representation playground filled with mask tokens + positional encodings
        pred_tokens = self.mask_token.expand(B, target_idx.shape[1], -1)
        # Add target positional information to the mask tokens
        target_pos = self.pos_embed.expand(B, -1, -1)
        target_pos_gathered = torch.gather(target_pos, dim=1, index=target_idx.unsqueeze(-1).expand(-1, -1, context_feats.shape[-1]))
        pred_tokens = pred_tokens + target_pos_gathered
        
        # Combine context features and target mask tokens to predict
        x = torch.cat([context_feats, pred_tokens], dim=1)
        
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        
        # Return only the predictions corresponding to the target positions
        return x[:, context_feats.shape[1]:]

## 4. Helper Function: Block Masking Strategy
Simple programmatic strategy to split the 7x7 patch map into a context block and a target block.


In [13]:
def get_block_masks(batch_size, num_patches=49, grid_size=7):
    # For a simple MNIST execution:
    # We designate a static target region (e.g., center 3x3 patch grid) 
    # and a context region (the surrounding frame patches) to keep execution clean.
    
    all_indices = list(range(num_patches))
    
    # Target: Center region of the 7x7 grid
    target_idx_list = []
    for r in range(2, 5):
        for c in range(2, 5):
            target_idx_list.append(r * grid_size + c)
            
    context_idx_list = [i for i in all_indices if i not in target_idx_list]
    
    target_idx = torch.tensor(target_idx_list, dtype=torch.long).unsqueeze(0).expand(batch_size, -1).to(device)
    context_idx = torch.tensor(context_idx_list, dtype=torch.long).unsqueeze(0).expand(batch_size, -1).to(device)
    
    return context_idx, target_idx


# ## 5. JEPA Pre-training Engine

In [14]:

# Initialize systems
context_encoder = JEPAEncoder().to(device)
target_encoder = copy.deepcopy(context_encoder).to(device)
# Target encoder weights are frozen from explicit gradients
for p in target_encoder.parameters():
    p.requires_grad = False

predictor = JEPAPredictor().to(device)

optimizer = optim.AdamW(list(context_encoder.parameters()) + list(predictor.parameters()), lr=LR)
criterion = nn.MSELoss()

print("Beginning Self-Supervised JEPA Training Phase...")

for epoch in range(1, EPOCHS + 1):
    context_encoder.train()
    predictor.train()
    total_loss = 0
    
    for i, (imgs, _) in enumerate(train_loader):
        imgs = imgs.to(device)
        B = imgs.shape[0]
        
        # Get patch masks
        context_idx, target_idx = get_block_masks(B)
        
        # 1. Target encoder processes full unaltered image
        with torch.no_grad():
            target_feats_full = target_encoder(imgs)
            # Gather targets corresponding strictly to the target block indices
            t_idx = target_idx.unsqueeze(-1).expand(-1, -1, target_feats_full.shape[-1])
            target_feats = torch.gather(target_feats_full, dim=1, index=t_idx)
            
        # 2. Context encoder processes only visible partial patches
        context_feats = context_encoder(imgs, patch_indices=context_idx)
        
        # 3. Predictor tries to guess target latents
        predicted_target_feats = predictor(context_feats, context_idx, target_idx)
        
        # 4. Loss evaluated in embedding space
        loss = criterion(predicted_target_feats, target_feats)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 5. EMA Update for the Target Encoder
        with torch.no_grad():
            for param_q, param_k in zip(context_encoder.parameters(), target_encoder.parameters()):
                param_k.data = param_k.data * EMA_DECAY + param_q.data * (1.0 - EMA_DECAY)
                
        total_loss += loss.item()
        
    print(f"Epoch [{epoch}/{EPOCHS}] | Latent Space MSE Loss: {total_loss / len(train_loader):.4f}")

Beginning Self-Supervised JEPA Training Phase...
Epoch [1/5] | Latent Space MSE Loss: 0.4578
Epoch [2/5] | Latent Space MSE Loss: 0.3748
Epoch [3/5] | Latent Space MSE Loss: 0.3438
Epoch [4/5] | Latent Space MSE Loss: 0.3272
Epoch [5/5] | Latent Space MSE Loss: 0.3120


## 6. Downstream Evaluation: Linear Probing
We freeze the trained Context Encoder, extract complete latents, and train a simple Linear Classifer layer to decode the digits.


In [15]:

class LinearProbe(nn.Module):
    def __init__(self, embed_dim=64, num_classes=10):
        super().__init__()
        self.linear = nn.Linear(embed_dim, num_classes)
        
    def forward(self, x):
        # Average pool across the sequence dimension
        x = x.mean(dim=1)
        return self.linear(x)

probe = LinearProbe().to(device)
probe_optimizer = optim.AdamW(probe.parameters(), lr=1e-3)
probe_criterion = nn.CrossEntropyLoss()

# Freeze Context Encoder explicitly
context_encoder.eval()
for p in context_encoder.parameters():
    p.requires_grad = False

print("\nTraining Downstream Linear Probe Classifier...")
for epoch in range(1, 3):  # Quick conversion pass
    probe.train()
    correct = 0
    total = 0
    
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        
        with torch.no_grad():
            # Process complete image patches
            features = context_encoder(imgs)
            
        outputs = probe(features)
        loss = probe_criterion(outputs, labels)
        
        probe_optimizer.zero_grad()
        loss.backward()
        probe_optimizer.step()
        
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    print(f"Probe Epoch [{epoch}/2] | Train Accuracy: {100. * correct / total:.2f}%")


Training Downstream Linear Probe Classifier...
Probe Epoch [1/2] | Train Accuracy: 77.67%
Probe Epoch [2/2] | Train Accuracy: 85.14%


In [16]:
## 7. Final Test Verification

In [17]:
probe.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        features = context_encoder(imgs)
        outputs = probe(features)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

print(f"\nFinal Downstream Classification Accuracy on Test Set: {100. * test_correct / test_total:.2f}%")


Final Downstream Classification Accuracy on Test Set: 85.62%
